# 03 – Classify for RiverEggCode
**Purpose:** ....


**Workflow:**

3.0 | Imports

3.1 | Configuration cell

3.2 | 

&nbsp;&nbsp;&nbsp;&nbsp;3.2.1 | Classify Reaches

&nbsp;&nbsp;&nbsp;&nbsp;3.2.2 | Aggregate Reaches into Eggs


---
## 3.0 | Imports

In [ ]:
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import sys

sys.path.append(os.path.join("..", "src"))
from _0_config_paths import DATA_PROCESSED, DATA_OUTPUT
from _3_2_1_classifier import classify_all
from _3_2_2_egg_code   import build_all_eggs, egg_to_dataframe, egg_to_string

---
## 3.1 | Configuration cell

In [ ]:
# Input: joined dataset from Notebook 02 
IN_JOINED = os.path.join(DATA_PROCESSED, "sword_naryn_GRITv1.0_joined.gpkg")

# Outputs
OUT_CLASSIFIED = os.path.join(DATA_PROCESSED, "sword_naryn_classified.gpkg")
OUT_EGGS_CSV   = os.path.join(DATA_OUTPUT, "naryn_eggs.csv")

print("Configuration set:")
print(f"Input : {IN_JOINED}")
print(f"Output : {OUT_CLASSIFIED}")
print(f"Eggs : {OUT_EGGS_CSV}")

---
## 3.2 |
### 3.2.1 | Classify Reaches

Load joined dataset from notebook 02 and classify each SWORD reach individualy. Every SWORD reach gets its own egg_ columns based on its own measured values (slope, n_chan_mod, facc etc.):

In [ ]:
gdf = gpd.read_file(IN_JOINED)

print(f"Reaches loaded: {len(gdf)}")
print(f"CRS : {gdf.crs}")


In [ ]:
gdf_classified = classify_all(gdf)

# Show result
egg_cols = [c for c in gdf_classified.columns if c.startswith("egg_")]
print("\nClassified columns:", egg_cols)
gdf_classified[["reach_id", "dist_out"] + egg_cols].head(10)

# Save classified reaches (one row per SWORD reach, with egg_ columns added):
os.makedirs(DATA_PROCESSED, exist_ok=True)
gdf_classified.to_file(OUT_CLASSIFIED, driver="GPKG")
print(f"Saved: {OUT_CLASSIFIED}")

### 3.2.2 | Aggregate reaches into eggs
One Egg per global_id_GRITv1.0. Reaches sorted upstream → downstream via dist_out.

In [ ]:
eggs = build_all_eggs(gdf_classified)

egg_df = egg_to_dataframe(eggs)

print(f"Total Eggs: {egg_df['global_id'].nunique()}")
print(f"Total Reaches: {len(egg_df)}")
print(f"\nReaches per Egg (distribution):")
print(egg_df.groupby("global_id")["reach_id"].count().describe())
print(f"\nSample:")
egg_df.head(15)

# Inspecting a single Egg:
EGG_INDEX = 0

print(egg_to_string(eggs[EGG_INDEX]))

Export Eggs as .csv:

In [ ]:
os.makedirs(DATA_OUTPUT, exist_ok=True)
egg_df.to_csv(OUT_EGGS_CSV, index=False)
print(f"Saved: {OUT_EGGS_CSV}")